# Transform Circuits Data

1. Read bronze `circuits` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`circuitId` → `circuit_id`, `circuitName` → `circuit_name`)
1. Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)
1. Filter out rows where `circuit_id` is null (business key validation)
1. Remove duplicate records
1. Transform values of columns `circuit_name` and `locality` to Title Case
1. Write the transformed data to silver `circuits` table

In [0]:
%run ../common/01_Environmnet_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

In [0]:
circuits_df = spark.read.table(bronze_table)
display(circuits_df)

In [0]:
from pyspark.sql import functions as F

circuits_drop_df = circuits_df.drop(F.col("url"))
display(circuits_drop_df)

In [0]:
circuits_renamed_col_df = (
    circuits_drop_df
        .withColumnsRenamed({
            "circuitId":"circuit_id",
            "circuitName":"circuit_name",
            "lat":"latitude",
            "long":"longitude"
        })
    ) 

circuits_renamed_col_df.display()

In [0]:
circuits_drop_null_df = (
    circuits_renamed_col_df.na.drop(subset=['circuit_id'])
)
circuits_drop_null_df.display()

In [0]:
circuits_drop_duplicates_df = circuits_drop_null_df.dropDuplicates(['circuit_id'])
circuits_drop_duplicates_df.display()

In [0]:
circuits_col_title_cap = (
    circuits_drop_duplicates_df
        .withColumns({
            "circuit_id" : F.initcap(F.col("circuit_id")),
            "locality" : F.initcap(F.col("locality"))
        })
)
circuits_col_title_cap.display()

In [0]:
(
circuits_col_title_cap
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)


In [0]:
%sql

select * from formula1.silver.circuits